# Week 12 Lab — Big Data Tools: SQL · APIs · PySpark

**Name:** NIYIBIZI IKUZWE Benjamin  
**Student ID:** ____________________  
**Program:** ____________________

**Course:** Introduction to Big Data Analytics — AUCA  
**Dataset:** `week12_transactions.xlsx` (2,000 transactions).

> Before running: upload this notebook and `week12_transactions.xlsx` to the same folder. For the API offline fallback, also place `week12_api_sample.json` beside the notebook. The lab handout names a CSV; this notebook reads the supplied Excel workbook and creates the equivalent CSV for SQL/Spark.

In [ ]:
!pip -q install pyspark openpyxl

In [ ]:
import pandas as pd
import sqlite3
import json
import requests
from pathlib import Path

# Load the supplied Excel data and create the CSV expected by the lab.
DATA_FILE = Path("week12_transactions.xlsx")
if not DATA_FILE.exists():
    raise FileNotFoundError("Place week12_transactions.xlsx next to this notebook.")
df = pd.read_excel(DATA_FILE)
required = {"txn_id", "date", "hour", "district", "txn_type", "amount_rwf", "flagged"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")
df.to_csv("week12_transactions.csv", index=False)
print("Rows:", len(df))
print("Total value (RWF):", f"{df['amount_rwf'].sum():,}")
df.head()

## Part 1 — SQL with Python (12 marks)
The following queries use SQLite and `pd.read_sql_query`, as required.

In [ ]:
conn = sqlite3.connect("momo.db")
df.to_sql("transactions", conn, if_exists="replace", index=False)

### Exercise 1.1 — SELECT / WHERE / ORDER BY
(a) Five largest `send` transactions above 100,000 RWF. (b) Count flagged transactions.

In [ ]:
q1a = """
SELECT txn_id, district, amount_rwf
FROM transactions
WHERE txn_type = 'send' AND amount_rwf > 100000
ORDER BY amount_rwf DESC, txn_id ASC
LIMIT 5;
"""
print(pd.read_sql_query(q1a, conn).to_string(index=False))
q1b = "SELECT COUNT(*) AS flagged_transactions FROM transactions WHERE flagged = 1;"
print(pd.read_sql_query(q1b, conn).to_string(index=False))

### Exercise 1.2 — GROUP BY
(a) Count, sum and average amount by transaction type, ordered by total descending. (b) Total amount by district.

In [ ]:
q2a = """
SELECT txn_type,
       COUNT(*) AS transaction_count,
       SUM(amount_rwf) AS total_amount_rwf,
       ROUND(AVG(amount_rwf), 2) AS average_amount_rwf
FROM transactions
GROUP BY txn_type
ORDER BY total_amount_rwf DESC;
"""
print(pd.read_sql_query(q2a, conn).to_string(index=False))
q2b = """
SELECT district, SUM(amount_rwf) AS total_amount_rwf
FROM transactions
GROUP BY district
ORDER BY total_amount_rwf DESC;
"""
print(pd.read_sql_query(q2b, conn).to_string(index=False))

### Exercise 1.3 — The busiest hour
(a) SQL counts by hour, busiest first. (b) Pandas independently computes the same counts.

In [ ]:
q3a = """
SELECT hour, COUNT(*) AS transaction_count
FROM transactions
GROUP BY hour
ORDER BY transaction_count DESC, hour ASC;
"""
sql_hours = pd.read_sql_query(q3a, conn)
print("SQL results:")
print(sql_hours.to_string(index=False))
pandas_hours = (df.groupby("hour").size().rename("transaction_count")
                .reset_index().sort_values(["transaction_count", "hour"], ascending=[False, True]).reset_index(drop=True))
print("\nPandas results:")
print(pandas_hours.to_string(index=False))
assert sql_hours.equals(pandas_hours), "SQL and Pandas results do not agree"
print("\nConfirmed: SQL and Pandas agree.")

## Part 2 — APIs & JSON (10 marks)

### Exercise 2.1 — JSON by hand
The handout refers to a JSON string in the starter notebook, but that starter notebook / JSON string was not among the uploaded files. This cell accepts the provided JSON sample if it is supplied. It supports a simple structure with `city` and `readings`; adjust the two key names only if the instructor's sample uses different names.

In [ ]:
sample_path = Path("week12_api_sample.json")
if sample_path.exists():
    with open(sample_path, "r", encoding="utf-8") as f:
        sample = json.load(f)  # equivalent to json.loads(f.read())
    city = sample.get("city", "Not specified")
    readings = sample.get("readings", [])
    values = [float(x.get("value", x) if isinstance(x, dict) else x) for x in readings]
    print("City:", city)
    print("Number of readings:", len(values))
    print("Average reading:", round(sum(values) / len(values), 2) if values else "No readings")
else:
    print("week12_api_sample.json not supplied. Add the instructor's sample file to run Exercise 2.1.")

### Exercise 2.2 — Call a real API
Open-Meteo provides a free forecast endpoint. If the live request fails, the code tries the offline sample file. The expected fields are `hourly.time` and `hourly.temperature_2m`.

In [ ]:
api_data = None
url = "https://api.open-meteo.com/v1/forecast"
params = {"latitude": -1.9441, "longitude": 30.0619,
          "hourly": "temperature_2m", "forecast_days": 1,
          "timezone": "Africa/Kigali"}
try:
    response = requests.get(url, params=params, timeout=15)
    response.raise_for_status()
    api_data = response.json()
    print("Source: live Open-Meteo API")
except Exception as err:
    print("Live API unavailable:", err)
    if sample_path.exists():
        with open(sample_path, "r", encoding="utf-8") as f:
            api_data = json.load(f)
        print("Source: offline sample JSON")
    else:
        print("No offline sample file found. Re-run this cell when internet is available or add the sample JSON.")

if api_data is not None:
    # Open-Meteo response structure
    hourly = api_data.get("hourly", api_data)
    times = hourly.get("time", [])
    temps = hourly.get("temperature_2m", hourly.get("temp_c", []))
    if temps and times:
        hottest_index = temps.index(max(temps))
        print("Number of hours:", len(temps))
        print("Maximum temperature (°C):", max(temps))
        print("Minimum temperature (°C):", min(temps))
        print("Time of hottest hour:", times[hottest_index])
    else:
        print("The JSON does not contain the expected hourly time and temperature lists.")

### Exercise 2.3 — API data meets your toolbox
Build a DataFrame, describe temperatures and count hours at or above 25°C.

In [ ]:
if api_data is not None:
    hourly = api_data.get("hourly", api_data)
    times = hourly.get("time", [])
    temps = hourly.get("temperature_2m", hourly.get("temp_c", []))
    if times and temps and len(times) == len(temps):
        weather_df = pd.DataFrame({"time": times, "temp_c": temps})
        display(weather_df.describe())
        print("Hours at or above 25°C:", int((weather_df["temp_c"] >= 25).sum()))
        display(weather_df.head())
    else:
        print("Cannot build DataFrame: expected matching time and temperature lists are missing.")
else:
    print("Run Exercise 2.2 successfully before Exercise 2.3.")

## Part 3 — PySpark (10 marks)
Start a local Spark session, load the CSV, and repeat the relevant aggregations.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, count
spark = SparkSession.builder.appName("Week12BigDataLab").master("local[*]").getOrCreate()
sdf = spark.read.csv("week12_transactions.csv", header=True, inferSchema=True)
print("Row count:", sdf.count())
sdf.printSchema()
sdf.show(5, truncate=False)

### Exercise 3.2 — Filter

In [ ]:
print("Transactions above 100,000 RWF:", sdf.filter(col("amount_rwf") > 100000).count())
sdf.filter(col("flagged") == 1).show(5, truncate=False)

### Exercise 3.3 — groupBy, fourth reunion

In [ ]:
sdf.groupBy("txn_type").agg(
    count("*").alias("transaction_count"),
    spark_sum("amount_rwf").alias("total_amount_rwf")
).orderBy(col("total_amount_rwf").desc()).show()

district_totals = (sdf.groupBy("district")
                   .agg(spark_sum("amount_rwf").alias("total_amount_rwf"))
                   .toPandas().sort_values("total_amount_rwf", ascending=False))
print(district_totals.to_string(index=False))
# The Spark totals should match the SQL totals in Part 1, because both use the same source rows and grouping rules.

### Bonus — Spark speaks SQL too

In [ ]:
sdf.createOrReplaceTempView("transactions")
spark_sql = spark.sql(q2a)
spark_sql.show()
# The same SQL is powerful because SQL provides a familiar, portable query language;
# Spark can distribute the work across many machines while SQLite can run it locally.
# The query logic can therefore be reused as data size and computing infrastructure grow.

## Part 4 — Choose Your Weapon (3 marks)

1. **SQL** — The data is already in PostgreSQL, so SQL is the direct tool for grouping records by district and month inside the database.
2. **Pandas** — 40,000 rows are manageable on a typical computer, and Pandas supports cleaning, chart preparation and regression-model workflows.
3. **Spark** — 40 TB on an existing cluster requires distributed processing; Spark is designed to process data across cluster machines.
4. **API** — A weather API supplies current forecast data automatically, without requiring someone to upload a file.

## Part 5 — Final-Project Proposal (required)

**Proposed title: Predicting Potentially Flagged Mobile-Money Transactions in Rwanda.**

Our group will investigate whether transaction characteristics can help identify mobile-money transactions that are flagged for review. The research question is: *Can transaction hour, district, transaction type and transaction amount be used to classify whether a mobile-money transaction is flagged?* We will use the supplied November 2026 mobile-money transaction dataset, which contains 2,000 records and fields including transaction ID, date, hour, district, transaction type, amount and flagged status. The dataset is suitable for an initial study and meets the minimum requirement of 500 rows. This topic matters because early identification of unusual transactions can help financial-service providers prioritize reviews and strengthen transaction monitoring. We plan to use a **classification** model because the target variable, `flagged`, has two classes (0 = not flagged and 1 = flagged). Before modelling, we will inspect class balance, clean and encode categorical variables, split the data into training and testing sets, and evaluate performance using precision, recall, F1-score and a confusion matrix; accuracy alone may be misleading if flagged cases are uncommon.

## Submission checklist

- [ ] Save notebook as `Week12_NIYIBIZI_IKUZWE_Benjamin.ipynb`.
- [ ] Run all cells from top to bottom.
- [ ] Confirm SQL and Pandas hour counts agree.
- [ ] Confirm Spark totals match SQL totals.
- [ ] Run API cells live or provide the instructor's offline sample JSON.
- [ ] Capture screenshots of the required outputs.
- [ ] Submit the completed notebook and group proposal.

**Important data note:** the uploaded lab handout says the source is a CSV, but the uploaded dataset is an Excel workbook. This notebook converts it to the expected CSV format automatically. The instructor's `week12_api_sample.json` and starter notebook were not uploaded, so the handout's exact Exercise 2.1 JSON values cannot be calculated until that sample is provided.